# Tuned against untuned: is the search worth it?

One row is one client the bank called during a deposit campaign; `y` is `yes` when the client subscribed. The campaign lead: "A colleague spent a day tuning last year's model. Was that a good use of a day?"

## The starter

The cell applies D01 5.5's decision (`duration` is dropped: it is known only after the call), splits 80 / 20 with `stratify=y, random_state=42`, and defines `pre`, a `ColumnTransformer` that one-hot encodes the text columns and passes the numbers through. Expect 36,168 training and 9,043 test clients, 11.7% subscribers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("data/bank_marketing.csv")
df = df.rename(columns={"deposit": "y", "day": "day_of_week"})    # one set of names for every download route
for col in ["job", "education", "contact", "poutcome"]:
    df[col] = df[col].fillna("unknown")
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y", "duration"])                             # duration is known only after the call
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
print(len(X_train), "training clients,", len(X_test), "test clients,", f"{y.mean():.3f} subscribed")

36168 training clients, 9043 test clients, 0.117 subscribed


## Step 1 · Three folds

`cv = StratifiedKFold(3, shuffle=True, random_state=42)`. Three folds keep the search short on the course VM; the demonstration used five.

In [2]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(3, shuffle=True, random_state=42)

## Step 2 · The untuned boosted model

`cross_val_score(make_pipeline(pre, HistGradientBoostingClassifier(random_state=42)), X_train, y_train, cv=cv, scoring="roc_auc").mean()`.

In [3]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

model = make_pipeline(clone(pre), HistGradientBoostingClassifier(random_state=42))
untuned_cv = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
print(f"untuned, 3-fold CV ROC AUC {untuned_cv:.3f}")

untuned, 3-fold CV ROC AUC 0.795


## Step 3 · A random search with 15 candidates

```python
P = "histgradientboostingclassifier__"
dist = {P + "learning_rate": loguniform(0.01, 0.5), P + "max_leaf_nodes": randint(7, 128),
        P + "min_samples_leaf": randint(5, 200), P + "l2_regularization": loguniform(1e-3, 10)}
search = RandomizedSearchCV(model, dist, n_iter=15, cv=cv, scoring="roc_auc",
                            random_state=42, n_jobs=-1).fit(X_train, y_train)
```

Print `best_params_` and `best_score_`. How many models did the search fit?

In [4]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

P = "histgradientboostingclassifier__"
dist = {P + "learning_rate": loguniform(0.01, 0.5), P + "max_leaf_nodes": randint(7, 128),
        P + "min_samples_leaf": randint(5, 200), P + "l2_regularization": loguniform(1e-3, 10)}
search = RandomizedSearchCV(model, dist, n_iter=15, cv=cv, scoring="roc_auc",
                            random_state=42, n_jobs=-1).fit(X_train, y_train)
bp = search.best_params_
best_params = (f"learning_rate {bp[P + 'learning_rate']:.3f}, max_leaf_nodes {bp[P + 'max_leaf_nodes']}, "
               f"min_samples_leaf {bp[P + 'min_samples_leaf']}, l2_regularization {bp[P + 'l2_regularization']:.3f}")
fits = len(search.cv_results_["params"]) * cv.get_n_splits()
print(best_params)
print(f"best CV ROC AUC {search.best_score_:.3f} | {fits} fits")

learning_rate 0.046, max_leaf_nodes 56, min_samples_leaf 108, l2_regularization 0.005
best CV ROC AUC 0.796 | 45 fits


## Step 4 · How wide is the range?

Sort `pd.DataFrame(search.cv_results_)` by `rank_test_score` and look at the best and worst `mean_test_score`.

In [5]:
ranked = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(f"best {ranked['mean_test_score'].iloc[0]:.3f} to worst {ranked['mean_test_score'].iloc[-1]:.3f}")
ranked[["rank_test_score", "mean_test_score", "std_test_score"]].round(3)

best 0.796 to worst 0.771


,rank_test_score,mean_test_score,std_test_score
13,1,0.796,0.004
12,2,0.795,0.004
14,3,0.795,0.004
7,4,0.795,0.005
6,5,0.794,0.004
8,6,0.793,0.005
9,7,0.792,0.004
2,8,0.792,0.003
1,9,0.791,0.006
5,10,0.790,0.004


## Step 5 · The test clients, once per model

Score the test clients with the defaults fitted on all training rows, and with `search.best_estimator_`.

In [6]:
untuned = clone(model).fit(X_train, y_train)
auc_untuned = round(roc_auc_score(y_test, untuned.predict_proba(X_test)[:, 1]), 3)
auc_tuned = round(roc_auc_score(y_test, search.best_estimator_.predict_proba(X_test)[:, 1]), 3)
print(f"test ROC AUC: untuned {auc_untuned}, tuned {auc_tuned}, gain {auc_tuned - auc_untuned:.3f}")

test ROC AUC: untuned 0.803, tuned 0.804, gain 0.001


## Step 6 · The same question for a decision tree

`DecisionTreeClassifier(random_state=42)`: its 5-fold CV score with the defaults, then a `GridSearchCV` over `max_depth` in `[3, 4, 5, 6, 8, 10]` and `min_samples_leaf` in `[20, 50, 100, 200]`. Use five folds here, as the demonstration did: `StratifiedKFold(5, shuffle=True, random_state=42)`.

In [7]:
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier

cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
tree = make_pipeline(clone(pre), DecisionTreeClassifier(random_state=42))
tree_cv = cross_validate(tree, X_train, y_train, cv=cv5, scoring="roc_auc", return_train_score=True)
print(f"defaults: {tree_cv['test_score'].mean():.3f} ± {tree_cv['test_score'].std():.3f} "
      f"(training {tree_cv['train_score'].mean():.3f})")
grid = GridSearchCV(tree, {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10],
                           "decisiontreeclassifier__min_samples_leaf": [20, 50, 100, 200]},
                    cv=cv5, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
tree_params = (f"max_depth {grid.best_params_['decisiontreeclassifier__max_depth']}, "
               f"min_samples_leaf {grid.best_params_['decisiontreeclassifier__min_samples_leaf']}")
tree_test = round(roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]), 3)
print(f"tuned: {tree_params}, CV {grid.best_score_:.3f}, test {tree_test}")

defaults: 0.615 ± 0.012 (training 1.000)


tuned: max_depth 10, min_samples_leaf 200, CV 0.749, test 0.759


## Step 7 · "Was the day of tuning worth it?"

For each model, the gain from tuning in validation ROC AUC, set against the spread across folds. Two sentences, one per model, with the numbers.

In [8]:
boost_spread = cross_val_score(model, X_train, y_train, cv=cv5, scoring="roc_auc").std()
tree_gain = grid.best_score_ - tree_cv["test_score"].mean()
print(f"Boosting: the search gained {auc_tuned - auc_untuned:.3f} in test ROC AUC, inside the "
      f"{boost_spread:.3f} the defaults move from fold to fold, so a day of tuning bought nothing we can measure.")
print(f"Decision tree: tuning gained {tree_gain:.3f} in validation ROC AUC, because its defaults grow until "
      f"every leaf is pure; for this model the search was worth the time.")

Boosting: the search gained 0.001 in test ROC AUC, inside the 0.003 the defaults move from fold to fold, so a day of tuning bought nothing we can measure.
Decision tree: tuning gained 0.134 in validation ROC AUC, because its defaults grow until every leaf is pure; for this model the search was worth the time.


## Check yourself

In [9]:
check = {
    "Step 2 · untuned 3-fold CV ROC AUC": round(untuned_cv, 3),
    "Step 3 · fits": fits,
    "Step 3 · best parameters": best_params,
    "Step 3 · best CV ROC AUC": round(search.best_score_, 3),
    "Step 4 · best to worst": f"{ranked['mean_test_score'].iloc[0]:.3f} to {ranked['mean_test_score'].iloc[-1]:.3f}",
    "Step 5 · test ROC AUC, untuned": auc_untuned,
    "Step 5 · test ROC AUC, tuned": auc_tuned,
    "Step 5 · gain": round(auc_tuned - auc_untuned, 3),
    "Step 6 · tree defaults, 5-fold": f"{tree_cv['test_score'].mean():.3f} ± {tree_cv['test_score'].std():.3f}",
    "Step 6 · tree defaults, training AUC": round(tree_cv["train_score"].mean(), 3),
    "Step 6 · tree best parameters": tree_params,
    "Step 6 · tree tuned CV": round(grid.best_score_, 3),
    "Step 6 · tree tuned test": tree_test,
    "Step 7 · boosting fold-to-fold spread": round(boost_spread, 3),
    "Step 7 · tree gain": round(tree_gain, 3),
}
for label, value in check.items():
    print(f"{label:<40} {value}")

Step 2 · untuned 3-fold CV ROC AUC       0.795
Step 3 · fits                            45
Step 3 · best parameters                 learning_rate 0.046, max_leaf_nodes 56, min_samples_leaf 108, l2_regularization 0.005
Step 3 · best CV ROC AUC                 0.796
Step 4 · best to worst                   0.796 to 0.771
Step 5 · test ROC AUC, untuned           0.803
Step 5 · test ROC AUC, tuned             0.804
Step 5 · gain                            0.001
Step 6 · tree defaults, 5-fold           0.615 ± 0.012
Step 6 · tree defaults, training AUC     1.0
Step 6 · tree best parameters            max_depth 10, min_samples_leaf 200
Step 6 · tree tuned CV                   0.749
Step 6 · tree tuned test                 0.759
Step 7 · boosting fold-to-fold spread    0.003
Step 7 · tree gain                       0.134
